In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from scipy.stats import ttest_ind
import yfinance as yf
import pickle
from scipy.stats import chi2

In [ ]:
# Il progetto era stato inizialmente pensato per usare metodi classici di loop. 
# Successivamente per la gestione di grossi database si è prefetito utilizzare i calcoli vettoriali di pandas che restano il modo più efficiente di gestire i big data
# Condivido ugualmente il codice perchi volesse effettuare dei confronti di effeicienza di memoria fra i due codici 

In [ ]:
def historical_series(ticker_list: str, period: str, interval:str): 
  
  """fornire: 1)lista tickers, 2) il timeframe (i.e 5d, 5mo, 5y, ytd, max) 3) l'intervallo dei dati (5m, 5h, 5d, 1wk)
  Scaricare i dati storici da YaoohFinance: 1) dare la lista [1,1] 2) intestazione colonna "Stock Ticker" """
  
  ticker_dict = {}
  for t in ticker_list:
    yf_tickers = yf.Ticker(t)
    hist_data = yf_tickers.history(period= period, interval= interval)
    ticker_dict[t] = hist_data["Adj Close"]
  return pd.DataFrame(data= ticker_dict)

In [ ]:
def log_return(ticker_list: list, series):
  """Creiamo una funzione che calcola in automatico i rendimenti delle serie storiche aggiungendo una colonna. Fornire: lista dei ticker e le serie corrispondenti"""
  
  rendimenti_dict = {}
  for tick in ticker_list:
    series_copy =series.copy()
    series_copy= np.log(series_copy[tick])-np.log(series_copy[tick]).shift(1)
    rendimenti_dict[f"R_{tick}"] = series_copy

  rendimenti_df = pd.DataFrame(rendimenti_dict)
  columns_name = [col for col in rendimenti_df.columns if "R_" in col] #selezioniamo solo le colonne con "R_" (sono i nostri rendimenti) nell'indice
  series = rendimenti_df[columns_name].dropna() #la prima riga è stata eliminata poichè valore NaN

  return series, columns_name #output: 1) la serie rendimenti log, 2) i nomi delle nuove colonne rendimenti (sono tutte R_"ticker titolo")

In [ ]:
def save_pickle(file_name: str):
  """Funzioni  per salvare e aprire i nostri dati salvati in formato pickle (per efficienza)"""

  with open("file_name.pkl","wb") as f:
   return pickle.dump(file_name, f)

def open_pickle(file_name: str):
  """Creiamo anche la funzione per aprire i nostri file time file_name pickle"""
  with open(file_name, "rb") as f:
   return pickle.load(f)

In [ ]:
def basics_stats(df_log_returns: pd.DataFrame):
  """Funzione per ottenere le statistiche per analisi di forma della distribuzione"""
  
  stats = df_log_returns.describe().T # Trasponiamo per avere i ticker sulle righe
  stats['skewness'] = df_log_returns.skew()
  stats['kurtosis'] = df_log_returns.kurt()

  """output la kurtosis, la skewness e delle stats di base per ogni titolo in forma di tabella"""
  return stats 

In [ ]:
def data_slicer(df_returns, event_date:str, pre_event_days=600, gap_days=90, post_event_days=90):
    """
    Divide i rendimenti in due periodi: Normalità (Pre-evento) e Distress (Post-evento)
    
        df_returns: DataFrame con log-returns, event_date (str): Data dell'annuncio ,
        pre_event_days: Lunghezza della finestra di stima, gap_days: Giorni di 'cuscinetto' prima dell'evento,
        post_event_days: Lunghezza della finestra di impatto
    """
   
    t_event = pd.to_datetime(event_date)
    if df_returns.index.tz is not None:
        t_event = t_event.tz_localize(df_returns.index.tz) #gestione timezone dati

    # time event e calcolo dei df
    start_normal = t_event - pd.Timedelta(days=pre_event_days)
    end_normal   = t_event - pd.Timedelta(days=gap_days)
    
    start_distress = t_event + pd.Timedelta(days=1)
    end_distress   = t_event + pd.Timedelta(days=post_event_days)

    # Slicing 
    df_normal = df_returns.loc[start_normal:end_normal].copy()
    df_distress = df_returns.loc[start_distress:end_distress].copy()

    return df_normal, df_distress

In [ ]:
def historical_Var(alphas: list, ticker_list, log_return):
  """Funzione per calcolare il VaR_storico. Fornire: livelli di significatività, nome colonne return e le serie dei rendimenti"""
  
  Var_hist = {}
  for tick in ticker_list:
    Var_hist[tick] = {}
    for a in alphas:
      var_hist = log_return[tick].quantile(a)
      Var_hist[tick][a] = var_hist

  """output: la matrice dei value at risk calcolato con il metodo storico"""
  return pd.DataFrame.from_dict(Var_hist) 

In [ ]:
def matrice_variazioni(var_normale, var_distress, alpha, ticker_list):

  """calcola la variazione % del Var/ES fra i due scenari scelti."""
  delta = {}
  for tick in ticker_list:
    delta[f"∆% {tick}"] = {}
    for a in alphas:
      delta_variazione = (var_normale.loc[a, tick] - var_distress.loc[a, tick])/(var_normale.loc[a, tick])

      delta[f"∆% {tick}"][a] = delta_variazione

  delta_df = pd.DataFrame.from_dict(data= delta)
  """output: DF con la variazione fra i due scenari"""
  return delta_df 

In [ ]:
def expected_shortfall(alphas, ticker_list, return_log, df_var):
  """Funzione per calcolare gli ExpectedShortfall x livelli alpha. Fornire: 1) lista tickers 2) gli alpha desiderati 3) la lista dei rendimenti 4) la matrice di var"""

  es_results = {}
  for tick in ticker_list:
    es_results[tick] = {}
    for a in alphas:
      es_series = return_log[return_log[tick] < df_var.loc[a, tick]]
      es_calcolo = np.mean(es_series[tick])

      es_results[tick][a] = es_calcolo

  matrice_ES = pd.DataFrame(data= es_results)

  """output: 1) matrice Expected Shortfall, 2) la serie delle eccezioni"""
  return  matrice_ES , es_series 

In [ ]:
def matrice_variazioni(var_normale, var_distress, alpha, ticker_list):

  """calcola la variazione % del Var/ES fra i due scenari scelti."""
  delta = {}
  for tick in ticker_list:
    delta[f"∆% {tick}"] = {}
    for a in alphas:
      delta_variazione = (var_normale.loc[a, tick] - var_distress.loc[a, tick])/(var_normale.loc[a, tick])

      delta[f"∆% {tick}"][a] = delta_variazione

  delta_df = pd.DataFrame.from_dict(data= delta)
  """output: DF con la variazione fra i due scenari"""
  return delta_df 

In [ ]:
def kupiec_test(log_return, matrice_variazione, ticker_list):
  lista = {}
  for tick in ticker_list:
    lista_violazioni = log_return[log_return[tick]<matrice_variazione.loc[0.01,tick]]
    lista[tick] = lista_violazioni[tick]

  lista_df = pd.concat(lista, axis=1)

  n_violazioni = lista_df.count(False)

  p_hat = n_violazioni/log_return.count()
  p = 0.01

  """test di kupiec"""

  p_value = {}
  for tick in ticker_list:
    LCpof = -2*((log_return[tick].count()-n_violazioni)*np.log((1-p)/(1-p_hat)) + (n_violazioni*np.log(p/p_hat)))

    p_value_formula = 1-chi2.cdf(LCpof[tick], df=1)
    p_value[tick] = p_value_formula
     #t-test (output: p-value)

  values = pd.DataFrame(p_value, index=[0]) #risolve "ValueError: If using all scalar values, you must pass an index"

  return values #output: p_values di signidicatività del test di kupiec 